In [ ]:
import workshop_setup


In [ ]:
# Checking the installation
!reinvent --help

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import mols2grid
import os


In [ ]:
global_parameters = """
run_type = "staged_learning"
device = "cpu"
tb_logdir = "tb_logs_sigma128_unusualmw"
json_out_config = "_stage1.json"
"""

prior_filename = os.path.join("priors/reinvent.prior")
agent_filename = prior_filename

parameters = f"""
[parameters]

prior_file = "{prior_filename}"
agent_file = "{agent_filename}"
summary_csv_prefix = "/nfs/home/myuecel3/Frontiers_MedChem/data/output/reinvent_rl/molwt"

batch_size = 64

use_checkpoint = false
"""


learning_strategy = """
[learning_strategy]

type = "dap"
sigma = 128
rate = 0.0001
"""

stages = """
[[stage]]

max_score = 1.0
min_steps = 5
max_steps = 10

chkpt_file = 'stage1.chkpt'

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"

params.smarts = [
    "[*;r8]",
    "[*;r9]",
    "[*;r10]",
    "[*;r11]",
    "[*;r12]",
    "[*;r13]",
    "[*;r14]",
    "[*;r15]",
    "[*;r16]",
    "[*;r17]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#7;!n]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#16;!s][C;!$(C(=[O,N])[N,O])][#16;!s]"
]
[[stage.scoring.component]]
[stage.scoring.component.MolecularWeight]

[[stage.scoring.component.MolecularWeight.endpoint]]
name = "Molecular weight"  # user chosen name for output
weight = 1  # weight to fine-tune the relevance of this component

transform.type = "double_sigmoid"
transform.high = 1200.0
transform.low = 700.0
transform.coef_div = 1200.0
transform.coef_si = 20.0
transform.coef_se = 20.0
"""



In [ ]:
config = global_parameters + parameters + learning_strategy + stages

toml_config_filename = "configs/reinvent_rl_config.toml"

with open(toml_config_filename, "w") as tf:
    tf.write(config)

In [ ]:
!python -m reinvent.Reinvent -l stage1.log $toml_config_filename

In [ ]:
!tensorboard  --logdir tb_logs_sigma128_unusualmw_0 --load_fast true

In [ ]:
global_parameters = """
run_type = "staged_learning"
device = "cpu"
tb_logdir = "tb_logs_numrotbond"
json_out_config = "_stage1.json"
"""

prior_filename = os.path.join("./priors/reinvent.prior")
agent_filename = prior_filename

parameters = f"""
[parameters]

prior_file = "{prior_filename}"
agent_file = "{agent_filename}"
summary_csv_prefix = "stage_numrotbond"

batch_size = 100

use_checkpoint = false
"""


learning_strategy = """
[learning_strategy]

type = "dap"
sigma = 128
rate = 0.0001

[diversity_filter]

type = "IdenticalMurckoScaffold" # IdenticalTopologicalScaffold,
                                 # ScaffoldSimilarity, PenalizeSameSmiles
bucket_size = 10                 # memory size in number of compounds
minscore = 0.4                   # only memorize if this threshold is exceeded
minsimilarity = 0.4              # minimum similarity for ScaffoldSimilarity
penalty_multiplier = 0.5         # penalty factor for PenalizeSameSmiles


"""

stage = """
[[stage]]

max_score = 1.0
min_steps = 50
max_steps = 300

chkpt_file = 'numrotbond_reversesig.chkpt'

[stage.scoring]
type = "geometric_mean"

######keep the alerts to avoid unwanted structures

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"

params.smarts = [
    "[*;r8]",
    "[*;r9]",
    "[*;r10]",
    "[*;r11]",
    "[*;r12]",
    "[*;r13]",
    "[*;r14]",
    "[*;r15]",
    "[*;r16]",
    "[*;r17]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#7;!n]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#16;!s][C;!$(C(=[O,N])[N,O])][#16;!s]"
]


[[stage.scoring.component]]
[stage.scoring.component.NumRotBond]

[[stage.scoring.component.NumRotBond.endpoint]]
name = "ROTB"
weight = 1
transform.type = "reverse_sigmoid"
transform.high = 20
transform.low = 0
transform.k = 0.5

"""

In [ ]:
config = global_parameters + parameters+ learning_strategy + stage
toml_config_filename = "configs/reward_numrotb.toml"
with open(toml_config_filename, 'w') as tf:
    tf.write(config)


In [ ]:
!python -m reinvent.Reinvent -l numrotb.log $toml_config_filename

In [ ]:
%tensorboard --logdir /content/REINVENT4/tb_logs_numrotbond --load_fast true

In [ ]:
global_sampling_parameters= """
run_type = "sampling"
device = "cpu"
json_out_config = "_sampling.json"
"""
parameters = """
[parameters]
model_file = "./numrotbond_reversesig.chkpt" ##
output_file = 'samplin_numrotbond.csv'
num_smiles = 1200
unique_molecules = true
randomize_smiles = true
"""

In [ ]:
samp_conf = global_sampling_parameters + parameters
with open("sampling_numrotbond.toml", "w") as tf:
    tf.write(samp_conf)

In [ ]:
!python -m reinvent.Reinvent -l samplin.log /content/REINVENT4/sampling_numrotbond.toml

In [ ]:

global_scoring_parameters = """
run_type = "scoring"
json_out_config = "_scoring.json"
"""

parameters = """
[parameters]
smiles_file = "samplin_numrotbond.csv"   #the path of sampled molecules
output_csv = "scoring_numrotbond.csv"
"""
scoring = """
[scoring]
type = "geometric_mean"

[[scoring.component]]
[scoring.component.GraphLength]
[[scoring.component.GraphLength.endpoint]]
name = "GraphLenght" #number of bonds in longest path
weight = 0.2
transform.type = "reverse_sigmoid"
transform.high = 40
transform.low = 20
transform.k = 0.5

######
#  Add scoring functions as many as you want. Define the weights for each score function and tranform them properly.
######



"""


In [ ]:
scoring_conf = global_scoring_parameters + parameters + scoring
with open("scoring_numrotbond.toml", "w") as tf:
    tf.write(scoring_conf)

In [ ]:
!python -m reinvent.Reinvent -l scoring.log /content/REINVENT4/scoring_numrotbond.toml

In [ ]:
scored_mols = pd.read_csv("/content/REINVENT4/scoring_numrotbond.csv")

# Convert SMILES to RDKit molecule objects for visualization
scored_mols['mols'] = [Chem.MolFromSmiles(smi) for smi in scored_mols["SMILES"]]

# Sort by score (best molecules first)
scored_mols = scored_mols.sort_values('Score', ascending=False).reset_index(drop=True)

# Interactive molecular grid
mols2grid.display(
    scored_mols,
    mol_col='mols',
    dpi=300,
    subset=['Score'],  # You can also add the scoring components, if you want to analyse them too.
    transform={
        "Score": lambda x: f"{x:.2f}" # add the other scoring components to print transformed values instead of long, raw digits
    },
    n_items_per_page=12,
    size=(200, 200),
)
